# A4: Sensitivity Analysis — Coop-DABC vs Coop-SAC
### Appendix response: Reviewer R2.2 (Degradation) + Imbalance Tolerance

## Purpose
Sweep the two parameters one-dimensionally and check that **Coop-DABC stays
superior to Coop-SAC regardless of the parameter setting**.

| Appendix | Fixed | Varied | Scenarios |
|----------|------|------|-------------|
| **A (Degradation)** | tol = 5.0 MW | deg_cost = 2.5 / 5.0 / 7.5 / 10.0 $/MWh | 4 |
| **B (Tolerance)**   | deg = 5.0 $/MWh | tol = 2.0 / 5.0 / 10.0 MW | 3 |
| excluding the shared baseline | | **6 scenarios total** | |

## Algorithms
- **Coop-DABC**: MILP expert CSV → BC imitation + TD3 RL
- **Coop-SAC**: pure RL (no expert needed)

## Implication
> "Coop-DABC consistently outperforms Coop-SAC under every parameter setting,
> and both models are robust to parameter changes."

## Execution order
| Step | What | Note |
|------|------|------|
| **Config** | always run first | |
| **Step 1** | 6 MILP runs → expert CSVs | ~15-30 min |
| **Step 2** | check expert CSVs | instant |
| **Step 3** | train Coop-DABC (6 × N seeds) | hours |
| **Step 4** | train Coop-SAC (6 × N seeds) | hours |
| **Step 5** | save run log | instant |
| **Step 6** | collect results, comparison table | instant |
| **Step 7** | plots and CSV export | instant |


In [ ]:
import subprocess, sys, os, time, json, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CODE_DIR    = os.path.abspath('../code')
RESULTS_DIR = os.path.abspath('../results/sensitivity')
EXPERT_DIR  = os.path.abspath('../data/processed/expert_actions')
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Scenario definitions ──────────────────────────────────────────────────
SCEN_DEG = [(2.5, 5.0), (5.0, 5.0), (7.5, 5.0), (10.0, 5.0)]   # Appendix A
SCEN_TOL = [(5.0, 2.0), (5.0, 5.0), (5.0, 10.0)]                # Appendix B
ALL_SCENARIOS = SCEN_DEG + [(d, t) for d, t in SCEN_TOL if (d, t) not in SCEN_DEG]

SEEDS = [0, 1, 2, 3, 4]   # 5 seeds (matches baseline)

def expert_csv_path(cost, tol):
    return os.path.join(EXPERT_DIR, f'Offline_Expert_Action_joint_deg{cost}_tol{tol}.csv')

def result_json_path(algo, cost, tol, seed):
    return os.path.join(RESULTS_DIR, f'result_{algo}_deg{cost}_tol{tol}_seed{seed}.json')

def run_cmd(args_list, label='', cwd=CODE_DIR):
    """Run a subprocess; capture output and show it only on failure (avoids flooding)."""
    t0 = time.time()
    print(f'[RUN] {label}', flush=True)
    proc = subprocess.run(
        [sys.executable] + [str(a) for a in args_list],
        cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    elapsed = time.time() - t0
    if proc.returncode == 0:
        print(f'  done in {elapsed:.0f}s', flush=True)
    else:
        print(f'  [FAILED exit {proc.returncode}]  {elapsed:.0f}s', flush=True)
        print(proc.stdout[-3000:])
    return proc.returncode == 0

print(f'CODE_DIR     : {CODE_DIR}')
print(f'RESULTS_DIR  : {RESULTS_DIR}')
print(f'Total scenarios: {len(ALL_SCENARIOS)}  /  SEEDS: {SEEDS}')
print('(5,5) reuses baseline results -> automatically skipped during training')
print()
print('Scenario list:')
for i, (d, t) in enumerate(ALL_SCENARIOS):
    tag = ' * baseline (shared)' if (d, t) == (5.0, 5.0) else ''
    print(f'  {i+1}. deg={d} $/MWh  tol={t} MW{tag}')

## Step 1: MILP optimization — generate expert CSVs (6 runs)

In [ ]:
milp_log = []
for cost, tol in ALL_SCENARIOS:
    csv_path = expert_csv_path(cost, tol)

    # ── Skip if the file already exists ───────────────────────────────────
    if os.path.exists(csv_path):
        df_chk = pd.read_csv(csv_path)
        print(f'[SKIP] deg={cost}  tol={tol}  ->  {os.path.basename(csv_path)} already exists ({len(df_chk)} rows)')
        milp_log.append({'deg_cost': cost, 'tol': tol, 'ok': True, 'skipped': True})
        continue

    ok = run_cmd(
        ['optimization/run_optimization.py', '--mode', 'joint',
         '--deg_cost', cost, '--tol', tol],
        label=f'MILP  deg={cost}$/MWh  tol={tol}MW'
    )
    milp_log.append({'deg_cost': cost, 'tol': tol, 'ok': ok, 'skipped': False})

print('\n=== MILP done ===')
for r in milp_log:
    flag = '(skipped)' if r.get('skipped') else ''
    print(f"  deg={r['deg_cost']}  tol={r['tol']}  {'OK' if r['ok'] else 'FAILED'}  {flag}")

## Step 2: Check expert CSVs

In [ ]:
print(f'{"deg":>6}  {"tol":>5}  {"rows":>7}  {"bid_mean":>9}  {"ope_mean":>9}  status')
print('-' * 60)
missing = []
for cost, tol in ALL_SCENARIOS:
    path = expert_csv_path(cost, tol)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'{cost:>6}  {tol:>5}  {len(df):>7}  '
              f'{df.Bid_Action.mean():>9.3f}  {df.Ope_Action.mean():>9.3f}  OK')
    else:
        print(f'{cost:>6}  {tol:>5}  MISSING')
        missing.append((cost, tol))

if missing:
    print(f'\nWARNING: {len(missing)} missing - run Step 1 first')
else:
    print('\nAll expert CSVs OK')

## Step 3: Train Coop-DABC (6 scenarios × N seeds)

In [ ]:
dabc_log = []
for cost, tol in ALL_SCENARIOS:
    csv = expert_csv_path(cost, tol)
    if not os.path.exists(csv):
        print(f'SKIP deg={cost} tol={tol}: expert CSV missing')
        continue
    for seed in SEEDS:
        rp = result_json_path('coop_bc', cost, tol, seed)
        if os.path.exists(rp):
            print(f'[SKIP] Coop-DABC  deg={cost} tol={tol} seed={seed} (result exists)')
            dabc_log.append({'algo': 'coop_dabc', 'deg_cost': cost, 'tol': tol, 'seed': seed, 'ok': True})
            continue
        ok = run_cmd(
            ['run_training.py', '--algo', 'coop_bc',
             '--deg_cost', cost, '--tol', tol, '--seed', seed,
             '--expert_csv', csv],
            label=f'Coop-DABC  deg={cost}  tol={tol}  seed={seed}'
        )
        dabc_log.append({'algo': 'coop_dabc', 'deg_cost': cost, 'tol': tol, 'seed': seed, 'ok': ok})

print('\n=== Coop-DABC done ===')
df_dabc = pd.DataFrame(dabc_log)
if not df_dabc.empty:
    print(df_dabc.groupby(['deg_cost', 'tol'])['ok'].all().unstack('tol').to_string())

## Step 4: Train Coop-SAC (6 scenarios × N seeds)

In [ ]:
sac_log = []
for cost, tol in ALL_SCENARIOS:
    for seed in SEEDS:
        rp = result_json_path('coop_sac', cost, tol, seed)
        if os.path.exists(rp):
            print(f'[SKIP] Coop-SAC  deg={cost} tol={tol} seed={seed} (result exists)')
            sac_log.append({'algo': 'coop_sac', 'deg_cost': cost, 'tol': tol, 'seed': seed, 'ok': True})
            continue
        ok = run_cmd(
            ['run_training.py', '--algo', 'coop_sac',
             '--deg_cost', cost, '--tol', tol, '--seed', seed],
            label=f'Coop-SAC  deg={cost}  tol={tol}  seed={seed}'
        )
        sac_log.append({'algo': 'coop_sac', 'deg_cost': cost, 'tol': tol, 'seed': seed, 'ok': ok})

print('\n=== Coop-SAC done ===')
df_sac = pd.DataFrame(sac_log)
if not df_sac.empty:
    print(df_sac.groupby(['deg_cost', 'tol'])['ok'].all().unstack('tol').to_string())

## Step 5: Save run log

In [ ]:
all_runs = dabc_log + sac_log
if all_runs:
    df_log = pd.DataFrame(all_runs)
    log_path = os.path.join(RESULTS_DIR, 'sensitivity_run_log.csv')
    df_log.to_csv(log_path, index=False)
    total, ok_cnt = len(df_log), df_log['ok'].sum()
    print(f'Total {total} runs  ok: {ok_cnt}  failed: {total - ok_cnt}')
    failed = df_log[~df_log['ok']]
    if not failed.empty:
        print('\nFailed runs:')
        print(failed.to_string(index=False))
else:
    print('No experiments were run')

## Step 6: Collect results and build the comparison table
Reads `results/sensitivity/result_*.json` after training and builds the DABC vs SAC comparison.

In [ ]:
# Collect result JSON files
result_files = glob.glob(os.path.join(RESULTS_DIR, 'result_*.json'))
if not result_files:
    print(f'No result files: {RESULTS_DIR}/result_*.json')
    print('Steps 3 and 4 must finish first.')
else:
    records = []
    for f in result_files:
        with open(f) as fp:
            records.append(json.load(fp))

    df_all = pd.DataFrame(records)

    # Keep only this experiment's scenarios & algorithms
    scen_set = set(ALL_SCENARIOS)
    df_all['scenario'] = list(zip(df_all['deg_cost'], df_all['tol']))
    df_exp = df_all[
        df_all['algo'].isin(['coop_bc', 'coop_sac']) &
        df_all['scenario'].apply(lambda s: s in scen_set)
    ].copy()

    # Average over seeds
    df_agg = df_exp.groupby(['algo', 'deg_cost', 'tol']).agg(
        mean_total = ('test_mean_total', 'mean'),
        std_total  = ('test_std_total',  'mean'),
        mean_bid   = ('test_mean_bid',   'mean'),
        mean_ope   = ('test_mean_ope',   'mean'),
        mean_deg   = ('test_mean_deg',   'mean'),
        n_seeds    = ('seed',            'count'),
    ).reset_index()

    # DABC vs SAC pivot
    d = df_agg[df_agg['algo'] == 'coop_bc'][['deg_cost', 'tol', 'mean_total', 'std_total']].copy()
    s = df_agg[df_agg['algo'] == 'coop_sac'][['deg_cost', 'tol', 'mean_total', 'std_total']].copy()
    d = d.rename(columns={'mean_total': 'DABC_mean', 'std_total': 'DABC_std'})
    s = s.rename(columns={'mean_total': 'SAC_mean',  'std_total': 'SAC_std'})

    df_cmp = d.merge(s, on=['deg_cost', 'tol'], how='outer')
    df_cmp['Gap (DABC-SAC)'] = df_cmp['DABC_mean'] - df_cmp['SAC_mean']
    df_cmp['DABC_wins'] = df_cmp['Gap (DABC-SAC)'] > 0

    # Split into Appendix A / B
    df_cmp['Appendix'] = df_cmp.apply(
        lambda r: 'A (deg)' if r['tol'] == 5.0 else 'B (tol)', axis=1
    )

    print('=== DABC vs Coop-SAC sensitivity comparison ===')
    cols = ['Appendix', 'deg_cost', 'tol', 'DABC_mean', 'DABC_std', 'SAC_mean', 'SAC_std', 'Gap (DABC-SAC)', 'DABC_wins']
    print(df_cmp[cols].sort_values(['Appendix', 'deg_cost', 'tol']).to_string(index=False))
    print(f'\nScenarios where DABC wins: {df_cmp["DABC_wins"].sum()} / {len(df_cmp)}')

## Step 7: Plots and CSV export

In [ ]:
if 'df_cmp' not in dir():
    print('Run Step 6 first.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    w = 0.35

    # ── Appendix A: degradation sensitivity (tol fixed at 5 MW, deg varies) ──
    #    every point with tol==5.0, including the (5,5) baseline
    df_a = df_cmp[df_cmp['tol'] == 5.0].sort_values('deg_cost')
    x = np.arange(len(df_a))
    axes[0].bar(x - w/2, df_a['DABC_mean'], w, yerr=df_a['DABC_std'],
                label='Coop-DABC', color='steelblue', capsize=4)
    axes[0].bar(x + w/2, df_a['SAC_mean'],  w, yerr=df_a['SAC_std'],
                label='Coop-SAC',  color='darkorange', alpha=0.85, capsize=4)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([f'${c}/MWh' for c in df_a['deg_cost']])
    axes[0].set_xlabel('Degradation Cost (tol = 5 MW fixed)')
    axes[0].set_ylabel('Mean Daily Total Revenue ($)')
    axes[0].set_title('Appendix A: Degradation Cost Sensitivity')
    if 5.0 in list(df_a['deg_cost']):
        axes[0].axvline(list(df_a['deg_cost']).index(5.0), ls='--', color='gray',
                        alpha=0.5, label='baseline (5,5)')
    axes[0].legend()

    # ── Appendix B: tolerance sensitivity (deg fixed at 5 $/MWh, tol varies) ─
    #    every point with deg_cost==5.0, including the (5,5) baseline
    df_b = df_cmp[df_cmp['deg_cost'] == 5.0].sort_values('tol')
    x = np.arange(len(df_b))
    axes[1].bar(x - w/2, df_b['DABC_mean'], w, yerr=df_b['DABC_std'],
                label='Coop-DABC', color='steelblue', capsize=4)
    axes[1].bar(x + w/2, df_b['SAC_mean'],  w, yerr=df_b['SAC_std'],
                label='Coop-SAC',  color='darkorange', alpha=0.85, capsize=4)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([f'{t} MW' for t in df_b['tol']])
    axes[1].set_xlabel('Imbalance Tolerance (deg = 5 $/MWh fixed)')
    axes[1].set_ylabel('Mean Daily Total Revenue ($)')
    axes[1].set_title('Appendix B: Imbalance Tolerance Sensitivity')
    if 5.0 in list(df_b['tol']):
        axes[1].axvline(list(df_b['tol']).index(5.0), ls='--', color='gray',
                        alpha=0.5, label='baseline (5,5)')
    axes[1].legend()

    plt.suptitle('Coop-DABC vs Coop-SAC: Parameter Sensitivity', fontsize=13)
    plt.tight_layout()

    fig_path = os.path.join(RESULTS_DIR, 'Sensitivity_DABC_vs_SAC.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()

    # Save CSV
    csv_path = os.path.join(RESULTS_DIR, 'Sensitivity_Summary.csv')
    df_cmp[cols].sort_values(['Appendix', 'deg_cost', 'tol']).to_csv(csv_path, index=False)

    print(f'Figure → {fig_path}')
    print(f'CSV    → {csv_path}')

---
## Reference: direct terminal commands

```bash
# (from code/)

# ── Appendix A: degradation sensitivity (tol fixed at 5.0) ───────────
python optimization/run_optimization.py --mode joint --deg_cost 2.5  --tol 5.0
python optimization/run_optimization.py --mode joint --deg_cost 5.0  --tol 5.0
python optimization/run_optimization.py --mode joint --deg_cost 7.5  --tol 5.0
python optimization/run_optimization.py --mode joint --deg_cost 10.0 --tol 5.0

python run_training.py --algo coop_bc  --deg_cost 2.5  --tol 5.0 --seed 0 --expert_csv ../data/processed/expert_actions/Offline_Expert_Action_joint_deg2.5_tol5.0.csv
python run_training.py --algo coop_sac --deg_cost 2.5  --tol 5.0 --seed 0

# ── Appendix B: tolerance sensitivity (deg fixed at 5.0) ─────────────
python optimization/run_optimization.py --mode joint --deg_cost 5.0 --tol 2.0
python optimization/run_optimization.py --mode joint --deg_cost 5.0 --tol 10.0

python run_training.py --algo coop_bc  --deg_cost 5.0 --tol 2.0  --seed 0 --expert_csv ../data/processed/expert_actions/Offline_Expert_Action_joint_deg5.0_tol2.0.csv
python run_training.py --algo coop_sac --deg_cost 5.0 --tol 2.0  --seed 0
```

### Scenario summary

| # | deg_cost | tol | Appendix | baseline? |
|---|----------|-----|----------|-----------|
| 1 | 2.5 | 5.0 | A | |
| 2 | **5.0** | **5.0** | A & B | **★** |
| 3 | 7.5 | 5.0 | A | |
| 4 | 10.0 | 5.0 | A | |
| 5 | 5.0 | 2.0 | B | |
| 6 | 5.0 | 10.0 | B | |
